# Fine-tuning the YOLOv11 instance segmentation model

Training notebook for the paper *A deep learning-enhanced PTV framework for
simultaneous translational and rotational tracking of rod-like particles in
non-Newtonian fluids*.

It downloads the annotated dataset from Roboflow and fine-tunes a
COCO-pretrained YOLOv11-small segmentation model on it. The resulting weights
are the ones shipped in `model/best.pt` and consumed by
`02_run_ptv_tracking.ipynb`.

**Dataset**:
[deep-learning-ptv-wolff-et-al](https://app.roboflow.com/particle-tracking-velocimetry/deep-learning-ptv-wolff-et-al)
— 673 images, one class (`Fiber`), instance-segmentation masks.

**Hardware**: training was run on a single NVIDIA RTX 4060 Ti (8 GB) and took
about 5.9 h for 40 epochs at 1024 x 1024. A CUDA-capable GPU is effectively
required; this notebook will not train in reasonable time on a CPU or on Apple
Silicon.


## 1. Dependencies


In [ ]:
# Pinned to the version used to train the published model. Newer 8.3.x
# releases also work, but segmentation output can differ between versions.
%pip install "ultralytics==8.3.9" "roboflow>=1.1.49" supervision

## 2. Configuration


In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
import os
from pathlib import Path

HOME = Path.cwd()

# --- Roboflow dataset --------------------------------------------------------
# Annotated dataset used to fine-tune the model:
#   https://app.roboflow.com/particle-tracking-velocimetry/deep-learning-ptv-wolff-et-al
WORKSPACE = "particle-tracking-velocimetry"
PROJECT   = "deep-learning-ptv-wolff-et-al"
VERSION   = 1          # set this to the dataset version shown on the page
FORMAT    = "yolov11"  # export format expected by the trainer

# The API key is read from the environment so it is never committed.
# Export it before launching Jupyter, or set it here temporarily:
#   export ROBOFLOW_API_KEY="your_key"
API_KEY = os.environ.get("ROBOFLOW_API_KEY")

# --- Training ----------------------------------------------------------------
BASE_MODEL = "yolo11s-seg.pt"  # COCO-pretrained YOLOv11 small, segmentation
EPOCHS     = 40
IMAGE_SIZE = 1024              # native resolution of the acquired frames

print(f"Working directory : {HOME}")
print(f"API key found     : {bool(API_KEY)}")

## 3. Dataset

The dataset is pulled with the Roboflow SDK, which needs an API key. If you
prefer not to use a key, download the export manually from the project page and
set `DATASET_LOCATION` to the extracted folder instead of running this cell.


In [ ]:
# =============================================================================
# DOWNLOAD THE ANNOTATED DATASET
# =============================================================================
from roboflow import Roboflow

if not API_KEY:
    raise RuntimeError(
        "ROBOFLOW_API_KEY is not set. Export it, or download the dataset "
        "manually from the project page and point DATASET_LOCATION at the "
        "extracted folder."
    )

roboflow_client = Roboflow(api_key=API_KEY)
project = roboflow_client.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download(FORMAT)

DATASET_LOCATION = Path(dataset.location)
print(f"Dataset downloaded to: {DATASET_LOCATION}")
print(f"Data config          : {DATASET_LOCATION / 'data.yaml'}")

## 4. Training


In [ ]:
# =============================================================================
# FINE-TUNE THE SEGMENTATION MODEL
# =============================================================================
# task=segment is required: passing task=detect with a *-seg model makes
# Ultralytics emit a warning and silently override it.
# Results, weights and diagnostic plots are written to runs/segment/train/.

%cd {HOME}

!yolo task=segment mode=train model={BASE_MODEL} data={DATASET_LOCATION}/data.yaml epochs={EPOCHS} imgsz={IMAGE_SIZE} plots=True

## 5. Diagnostics


In [ ]:
# =============================================================================
# TRAINING DIAGNOSTICS
# =============================================================================
from IPython.display import Image as IPyImage, display

TRAIN_DIR = HOME / "runs" / "segment" / "train"

for name, caption in (
    ("confusion_matrix.png", "Confusion matrix"),
    ("results.png",          "Loss and metric curves"),
    ("val_batch0_pred.jpg",  "Predictions on a validation batch"),
):
    path = TRAIN_DIR / name
    if path.exists():
        print(caption)
        display(IPyImage(filename=str(path), width=600))
    else:
        print(f"{caption}: not found at {path}")

## 6. Collect the weights


In [ ]:
# =============================================================================
# COLLECT THE TRAINED WEIGHTS
# =============================================================================
# The tracking notebook reads the weights from model/best.pt. Copy the best
# checkpoint of this run there to use it downstream.
import shutil

REPO_ROOT = HOME.parent if HOME.name == "notebooks" else HOME
source = HOME / "runs" / "segment" / "train" / "weights" / "best.pt"
target = REPO_ROOT / "model" / "best.pt"

if source.exists():
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
    print(f"Copied {source} -> {target}")
else:
    print(f"No checkpoint found at {source}; run the training cell first.")